# E21 CPU kit (Kaggle pinned image)

Harness modes of both E21 notebooks, real vLLM 0.19 / 0.32 parsing of every launch command, and the real FP8 / INT4 model files. Kit: `foysalemonshanto/q27-e21-cpu-kit` (q27x_e21_cpu_kit_5f0dafa3faab.bin).


In [ ]:
# E21 CPU kit on Kaggle (pinned image = the GPU notebooks'): harness modes, real-vLLM flag parsing, real model files
import hashlib, io, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
KIT_NAME, KIT_SHA = "q27x_e21_cpu_kit_5f0dafa3faab.bin", "5f0dafa3faabb0a0751057a219e5bc369fa17d3a6aa5315aca053d71be9fa5f4"
NBS = {"nb1": "Q27-XHIGH-E21_DUCKB_VLLM_NB1_INT4_S20.ipynb", "nb2": "Q27-XHIGH-E21_DUCKB_VLLM_NB2_FP8_SSM16_S16.ipynb", "nb1s": "Q27-XHIGH-E21_DUCKB_VLLM_NB1_INT4_S20_SMOKE.ipynb", "nb2s": "Q27-XHIGH-E21_DUCKB_VLLM_NB2_FP8_SSM16_S16_SMOKE.ipynb"}
RUNS = [["nb1_gateAB", "nb1", "gateAB"], ["nb1_wdhold", "nb1", "wdhold"], ["nb1_overrun", "nb1", "overrun"], ["nb1_kvoutage", "nb1", "kvoutage"], ["nb1_gpuheldB", "nb1", "gpuheldB"], ["nb1s_commit", "nb1s", "commit"]]
WH = "/kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3"
WH032 = "/kaggle/input/datasets/foysalemonshanto/vllm-0-32-0-cu13-wheelhouse"
FP8 = "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"
INT4 = "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-int4-redhat/pytorch/w4a16-g128/1"
OUT = Path("/kaggle/working"); K = Path("/tmp/e21_kit"); T0 = time.time()
REPORT = {"kit": KIT_NAME, "python": sys.version, "nb_sha": {}, "harness": {}, "flagcheck": {}, "models": {}}


def save():
    (OUT / "cpu_kit_report.json").write_text(json.dumps(REPORT, indent=1), encoding="utf-8")


def find(name, sha):
    for pat in (name, "*/" + name, "*/*/" + name, "*/*/*/" + name, "*/*/*/*/" + name):
        for f in sorted(Path("/kaggle/input").glob(pat)):
            if hashlib.sha256(f.read_bytes()).hexdigest() == sha:
                return f.read_bytes()
    raise FileNotFoundError(f"{name} ({sha[:12]}) not under /kaggle/input: attach foysalemonshanto/q27-e21-cpu-kit")


shutil.rmtree(K, ignore_errors=True)
with zipfile.ZipFile(io.BytesIO(find(KIT_NAME, KIT_SHA))) as z:
    z.extractall(K)
man = json.loads((K / "MANIFEST.json").read_text())
bad = [k for k, h in man.items() if hashlib.sha256((K / k).read_bytes()).hexdigest() != h]
assert not bad, bad
REPORT["nb_sha"] = {k: hashlib.sha256((K / "nb" / v).read_bytes()).hexdigest() for k, v in NBS.items()}
print("kit OK:", len(man), "files |", sys.version.split()[0], "|", REPORT["nb_sha"], flush=True)

# ---- 3. the real model folders (FP8 vs INT4): template / tokenizer files and encode equality
try:
    _m = {}
    for tag, d in (("fp8", FP8), ("int4", INT4)):
        _m[tag] = {f: (hashlib.sha256((Path(d) / f).read_bytes()).hexdigest()[:16] if (Path(d) / f).is_file() else None)
                   for f in ("chat_template.jinja", "tokenizer.json", "tokenizer_config.json", "generation_config.json", "config.json")}
    REPORT["models"]["files"] = _m
    _tc = json.loads((Path(FP8) / "tokenizer_config.json").read_text(encoding="utf-8"))
    _j8 = (Path(FP8) / "chat_template.jinja").read_text(encoding="utf-8") if (Path(FP8) / "chat_template.jinja").is_file() else None
    _j4 = (Path(INT4) / "chat_template.jinja").read_text(encoding="utf-8") if (Path(INT4) / "chat_template.jinja").is_file() else None
    REPORT["models"]["template_equal"] = {"fp8_jinja_eq_int4_jinja": _j8 == _j4 and _j8 is not None,
                                          "fp8_tokenizer_config_eq_jinja": _tc.get("chat_template") in (None, _j8)}
    from tokenizers import Tokenizer
    ta, tb = Tokenizer.from_file(FP8 + "/tokenizer.json"), Tokenizer.from_file(INT4 + "/tokenizer.json")
    _s = ["0 0 1 1 2\n3 4 5 6 7 8 9\n" * 40, "def transform(grid):\n    return [row[::-1] for row in grid]\n",
          "<think>\nLet me check the 3x3 block at (2, 5)...\n</think>\n\n", _j8 or ""]
    REPORT["models"]["encode_equal"] = all(ta.encode(x).ids == tb.encode(x).ids for x in _s)
    _q = json.loads((Path(INT4) / "config.json").read_text(encoding="utf-8")).get("quantization_config") or {}
    REPORT["models"]["int4_quant"] = {"method": _q.get("quant_method"), "format": _q.get("format"),
                                      "ignores_mtp": "re:^mtp.*" in (_q.get("ignore") or [])}
except Exception as e:
    REPORT["models"]["error"] = repr(e)[:500]
save()
print("models:", json.dumps(REPORT["models"])[:1500], flush=True)

# ---- the GPU notebooks' runtime pins (CELL 2): transformers 4.57.6 + huggingface-hub 0.36.2 for the harness
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", WH,
                    "transformers==4.57.6", "huggingface-hub==0.36.2", "tokenizers==0.22.2"], capture_output=True, text=True)
REPORT["pins"] = {"rc": r.returncode, "tail": (r.stdout + r.stderr)[-800:]}

# ---- 1. harness modes
ENV = dict(os.environ, PYTHONIOENCODING="utf-8", E14_FIXTURES_DIR=str(K / "fixtures"), E14_HARNESS_DIR=str(K / "e14src/harness"),
           Q27X_PAYLOAD_DIR=str(K / "nb/Build/payload"), E21_FAKE_SERVER=str(K / "e21/harness/fake_vllm_duck_server.py"))
for label, nbk, mode in RUNS:
    env = dict(ENV, E21_HARNESS_ROOT=f"/tmp/e21_h_{label}", E21_MODE=mode)
    t = time.time()
    try:
        r = subprocess.run([sys.executable, str(K / "e21/harness/harness_e21_nb.py"), str(K / "nb" / NBS[nbk])],
                           cwd=K / "e21/harness", env=env, capture_output=True, text=True, timeout=5400)
        out, rc = r.stdout + r.stderr, r.returncode
    except subprocess.TimeoutExpired as e:
        out, rc = (e.stdout or "") if isinstance(e.stdout, str) else "", "timeout"
    (OUT / f"harness_{label}.log").write_text(out, encoding="utf-8")
    lines = out.splitlines()
    REPORT["harness"][label] = {"rc": rc, "sec": round(time.time() - t), "ok": rc == 0 and "@@E21_HARNESS_OK@@" in out,
                                "summary": [x for x in lines if x.startswith("[e21")][-40:],
                                "fails": [x[:1500] for x in lines if "FAIL" in x][:40],
                                "tail": out[-3000:] if rc != 0 else ""}
    if mode == "commit":       # the launch commands this notebook makes (all halves, primary + fallback)
        for rp in sorted(Path(f"/tmp/e21_h_{label}").rglob("e21_report.json")):
            try:
                REPORT["flagcheck"].setdefault(nbk, {})["flags"] = json.loads(rp.read_text())["config"]["flags"]
            except Exception as e:
                REPORT["flagcheck"].setdefault(nbk, {})["flags_error"] = repr(e)[:300]
    save()
    print(label, "->", "ok" if REPORT["harness"][label]["ok"] else "FAIL", REPORT["harness"][label]["summary"][-6:],
          REPORT["harness"][label]["fails"][:6], flush=True)

REPORT["flagcheck"] = {"skipped": "another kit part runs the flag / tokenizer checks"}
_mods = REPORT["models"]
REPORT["all_ok"] = all(v["ok"] for v in REPORT["harness"].values())
REPORT["sec"] = round(time.time() - T0)
save()
print("ALL OK" if REPORT["all_ok"] else "NOT OK", "| total", REPORT["sec"], "s", flush=True)
